# TreeMMM budget-reallocation walkthrough

This reproducible example fits the pharma demo with seed 42, lands a committed touch increase under observed-support caps, and sweeps several budget levels. Values are model-outcome and aggregate-touch units; convert channels to common cost units upstream when per-touch costs differ.

In [ ]:
import treemmm
from treemmm.demo.datasets.pharma_brand import (
    generate_pharma_dataset,
    pharma_run_config,
)
from treemmm.mroi import reallocate, reallocate_curve

SEED = 42
CHANNEL = "rep_visits"
CAP_PERCENTILE = 95.0

In [ ]:
dataset = generate_pharma_dataset(
    n_customers=120,
    n_periods=12,
    random_state=SEED,
)
config = pharma_run_config(dataset)
config.n_optuna_trials = 2
config.random_state = SEED
result = treemmm.run(dataset.df, config)
print(result.summary())

In [ ]:
model = result.trained_models[-1]
feature_cols = config.columns.all_feature_cols()
X = result.prepared_data.df.loc[:, feature_cols].copy()

plan = reallocate(
    model,
    X,
    budget_delta_pct=25.0,
    channel=CHANNEL,
    cap_percentile=CAP_PERCENTILE,
)
{
    "predicted_incremental_outcome": plan.predicted_incremental_outcome,
    "predicted_lift_pct": plan.predicted_lift_pct,
    "unallocatable_fraction": plan.diagnostics.unallocatable_fraction,
}

In [ ]:
curve = reallocate_curve(
    model,
    X,
    budget_deltas=[10.0, 25.0, 50.0, 100.0],
    channel=CHANNEL,
    cap_percentile=CAP_PERCENTILE,
)
print(f"Largest fully allocatable delta: {curve.max_allocatable_delta}")
curve.table